# 02 — Tạo tập ứng viên để nhóm chấm nhãn
Mặc định chạy CPU Kaggle. Dùng model và wheel offline đã gắn vào input. Tạo embedding pretrained, **chưa fine-tune**. Grade để trống; không dùng tập test để lựa chọn model.

In [ ]:
DENSE_DEVICE = 'cpu'
EXTRAS = '[kaggle]'
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, importlib.util
if not Path('/kaggle/input').exists():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle, không train trên máy cá nhân.')
archives = list(Path('/kaggle/input').rglob('source.zip'))
source = Path('/kaggle/working/fashion-reco/source')
source.mkdir(parents=True, exist_ok=True)
if len(archives) == 1:
    workspace = archives[0].parent
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (source / member.filename).resolve().is_relative_to(source.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(source)
elif not archives:
    sources = [p.parent for p in Path('/kaggle/input').rglob('pyproject.toml') if (p.parent/'src/fashion_reco').is_dir()]
    if len(sources) != 1:
        raise RuntimeError('Thêm đúng một workspace có source.zip hoặc thư mục source/.')
    workspace = sources[0].parent
    shutil.copytree(sources[0], source, dirs_exist_ok=True)
else:
    raise RuntimeError('Có nhiều file source.zip trong input.')
if EXTRAS:
    # Import our source directly; never invoke a package build on Kaggle.
    requirements = {'numpy':'numpy>=1.26,<3', 'sklearn':'scikit-learn>=1.4,<2',
                    'yaml':'PyYAML>=6,<7', 'joblib':'joblib>=1.3,<2',
                    'rank_bm25':'rank-bm25>=0.2.2,<1', 'torch':'torch',
                    'sentence_transformers':'sentence-transformers>=3,<6',
                    'lightgbm':'lightgbm>=4,<5', 'pandas':'pandas'}
    wheels = list(workspace.rglob('rank_bm25-*.whl'))
    if importlib.util.find_spec('rank_bm25') is None and len(wheels)==1:
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',str(wheels[0])],check=True)
    missing = [value for name,value in requirements.items() if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--timeout','25','--retries','1',*missing],check=True)
    manifest_path = workspace/'offline-model-manifest.json'
    if manifest_path.exists():
        import json
        spec = importlib.util.spec_from_file_location('fashion_kaggle_runtime',source/'kaggle/run_experiment.py')
        runtime = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(runtime)
        runtime.OFFLINE_MODELS = json.loads(manifest_path.read_text(encoding='utf-8'))
        runtime.prepare_offline_models()
sys.path.insert(0, str(source / 'src'))
output = Path('/kaggle/working/fashion-reco/output')
if (workspace / 'previous-run').exists():
    shutil.copytree(workspace / 'previous-run', output, dirs_exist_ok=True)
os.environ['FASHION_CATALOG'] = str(workspace / 'catalog.json')
os.environ['FASHION_OUTPUT'] = str(output)
config_path = source / 'configs/experiment.yaml'
if EXTRAS:
    import torch, yaml
    device = globals().get('DENSE_DEVICE','cpu')
    if device not in ('cpu','cuda'):
        raise ValueError('Choose cpu or cuda explicitly')
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    config['dense'].update(device=device,batch_size=8 if device=='cpu' else 16)
    if device == 'cpu':
        torch.set_num_threads(4)
    elif not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable. Choose CPU or enable an actual GPU on Kaggle.')
    config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    print('Selected cloud device:',device)


In [ ]:
from fashion_reco.workflow import run_stage
pool_manifest = run_stage(config_path, 'label')
print('Số hàng cần chấm:', pool_manifest['row_count'])
print('Các loại món thiếu ứng viên:', pool_manifest['empty_slots'])

In [ ]:
import pandas as pd
labels = pd.read_csv(output / 'labeling/judgments.csv')
display(labels[['query_id','query','slot','product_name','price','grade']].head(20))
print('Lưu thư mục:', output / 'labeling')

## Chấm và đóng băng nhãn
0: không phù hợp; 1: một phần; 2: phù hợp; 3: rất phù hợp. Điền annotator; không xóa hàng. Hai người chấm test-rater-a/b độc lập; dùng adjudicate_test rồi giải quyết các ô bất đồng trong test-adjudicated.csv. Lưu labeling và artifact thành output/dataset private cho phiên sau. Không chạy label đè pool đã chấm.